In [10]:
import json
import glob
import re
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from macrogen import update_macro
from pathlib import Path
from natsort import natsorted

In [11]:
# RUN_LABEL = 'generate'
# DATASET = 'YTIDEOLOGY'
# DATASET_PATH = Path('../data/classification/yt_ideology')
# DATASET_INDEX = 'Unnamed: 0'
# RESULTS_DIR = Path('../results/') / RUN_LABEL / DATASET
# LABELS = ['Liberal', 'Neutral', 'Conservative']

In [12]:
RUN_LABEL = 'test'
DATASET = 'NEWSIDEOLOGY'
DATASET_PATH = Path('../data/classification/news_ideology')
DATASET_INDEX = 'ID'
RESULTS_DIR = Path('../results/') / RUN_LABEL / DATASET
LABELS = ['Liberal', 'Neutral', 'Conservative']

In [13]:
results = glob.glob(str(RESULTS_DIR / '**/20000_cands*/**/test-1000.json'), recursive=True)
results += glob.glob(str(RESULTS_DIR /  '**/0_cands*/**/test-1000.json'), recursive=True)
results

['../results/test/NEWSIDEOLOGY/test/12_shots/bertscore/20000_cands-deberta-large-mnli-recall/s0/LLAMA7B/test-1000.json',
 '../results/test/NEWSIDEOLOGY/test/12_shots/bertscore/20000_cands-deberta-large-mnli-recall/s0/MISTRAL/test-1000.json',
 '../results/test/NEWSIDEOLOGY/test/12_shots/bertscore/20000_cands-deberta-large-mnli-recall/s0/LLAMA13B/test-1000.json',
 '../results/test/NEWSIDEOLOGY/test/12_shots/bertscore/20000_cands-deberta-large-mnli-recall/s0/GPT4/test-1000.json']

In [14]:
def sanitize_prediction(x):
    if 'neutral' in x.lower():
        return 'Neutral'
    if 'liberal' in x.lower():
        return 'Liberal'
    if 'conservative' in x.lower():
        return 'Conservative'
    return 'Neutral'

In [15]:
test_set = pd.read_csv(DATASET_PATH / 'test.csv')
test_set['true'] = test_set['label'].map(lambda x : LABELS[x])
test_set = test_set.set_index(DATASET_INDEX)
test_set.head()

keys = []

for result in results:
    regex = f'results/{RUN_LABEL}/{DATASET}/test/(?P<num_shots>[0-9]+)?_shots/.*?/[0-9]+?_cands.*?/s0/(?P<llm_name>.*)?/test-1000.json'
    groups = re.search(
        re.compile(regex),
        result
    )

    llm_name = groups.group('llm_name')
    num_shots = groups.group('num_shots')

    print(llm_name, num_shots)

    with open(result) as f:
        js = json.load(f)

    key = f'{llm_name}_{num_shots}'
    keys.append(key)
    
    rows = []
    for res in js['results']:
        pred = sanitize_prediction(res['pred'])
        idx = res[DATASET_INDEX]
        rows.append({
            key: pred,
            'idx': idx
        })
    df = pd.DataFrame(rows).set_index('idx')
    test_set = test_set.join(df)

keys = natsorted(keys)
test_set = test_set.dropna(subset=keys)

LLAMA7B 12
MISTRAL 12
LLAMA13B 12
GPT4 12


In [16]:
def key_to_macro(k):
    llm_mapping = {
        'LLAMA7B': 'llamaseven',
        'LLAMA13B': 'llamathirteen',
        'GPT4': 'gpt',
        'MISTRAL': 'mistral'
    }
    shots_mapping = {
        '0': 'zero',
        '4': 'four',
        '8': 'eight',
        '12': 'twelve',
        '36': 'thirtysix',
        '125': 'onetwofive'
    }
    toks = k.split('_')
    return llm_mapping[toks[0]] + shots_mapping[toks[1]]

In [17]:
for key in keys:
    macro = f'{key_to_macro(key)}acc'
    print(key.ljust(15), '%.2f' % accuracy_score(test_set['true'], test_set[key]))
    # update_macro(macro, '%.2f' % accuracy_score(test_set['true'], test_set[key]))

GPT4_12         0.55
LLAMA7B_12      0.44
LLAMA13B_12     0.42
MISTRAL_12      0.48


In [18]:
for key in keys:
    macro = f'{key_to_macro(key)}'
    precision, recall, fscore, support = precision_recall_fscore_support(test_set['true'], test_set[key], labels=LABELS)
    for idx, label in enumerate(LABELS):
        key = f'{macro}{label.lower()}precision'
        # update_macro(key, '%.2f' % (precision[idx]))

        key = f'{macro}{label.lower()}recall'
        # update_macro(key, '%.2f' % (recall[idx]))